# Transmission Lines · Complex Wavenumber · Branch Cuts in Circuits

The same k that appears in quantum mechanics and optics lives in your PCB trace.
When k becomes complex, branch cuts are unavoidable.

In [ ]:
from sympy import *
from IPython.display import display, Markdown

init_printing(use_latex='mathjax')

def step(label, expr=None):
    display(Markdown(f'**{label}**'))
    if expr is not None:
        display(expr)

def section(title):
    display(Markdown(f'---\n## {title}'))

## §1 — Telegrapher's Equations: Maxwell on a Wire

In [ ]:
section('Telegrapher equations')

display(Markdown(r'''
A transmission line has distributed parameters per unit length:

| Symbol | Meaning | Units |
|---|---|---|
| $R'$ | resistance | Ω/m |
| $L'$ | inductance | H/m |
| $G'$ | conductance | S/m |
| $C'$ | capacitance | F/m |

Maxwell's equations on this line give the **telegrapher's equations**:
$$\frac{\partial V}{\partial z} = -(R' + i\omega L')\, I$$
$$\frac{\partial I}{\partial z} = -(G' + i\omega C')\, V$$
'''))

omega, z = symbols('omega z', real=True)
R, L, G, C = symbols("R' L' G' C'", positive=True)

# Combine into second order
Z_ser = R + I*omega*L   # series impedance per unit length
Y_shu = G + I*omega*C   # shunt admittance per unit length
step('Series impedance Z = R\' + iωL\' =', Z_ser)
step('Shunt admittance Y = G\' + iωC\' =', Y_shu)

# Propagation constant
gamma_sq = Z_ser * Y_shu
gamma = sqrt(gamma_sq)
step('Propagation constant γ² = ZY =', expand(gamma_sq))
step('γ = √(ZY) =', gamma)
step('γ = α + iβ  where α = attenuation [Np/m], β = phase constant [rad/m]')

## §2 — Complex Wavenumber: γ = α + iβ

In [ ]:
section('Complex wavenumber')

display(Markdown(r'''
The general solution to the telegrapher equations:
$$V(z) = V^+e^{-\gamma z} + V^-e^{+\gamma z}$$
$$I(z) = \frac{V^+}{Z_0}e^{-\gamma z} - \frac{V^-}{Z_0}e^{+\gamma z}$$

With $\gamma = \alpha + i\beta$:
$$V^+e^{-\gamma z} = V^+e^{-\alpha z}\cdot e^{-i\beta z}$$

- $e^{-\alpha z}$: **exponential decay** (attenuation)
- $e^{-i\beta z}$: **phase advance** (wavenumber = β)

This is identical to the optical field in a lossy fiber:
$$E(z) = E_0\, e^{-\alpha z/2}\, e^{i\beta z}$$
with complex refractive index $\tilde{n} = n_r + in_i$,
$\alpha = 2\omega n_i/c$, $\beta = \omega n_r/c$.
'''))

alpha_sym, beta_sym = symbols('alpha beta', positive=True)
gamma_sym = alpha_sym + I*beta_sym

# Phase velocity
v_phase = omega / beta_sym
step('Phase velocity v_p = ω/β =', v_phase)

# Wavelength on line
lam_line = 2*pi / beta_sym
step('Guided wavelength λ_g = 2π/β =', lam_line)

# Lossless case
beta_lossless = omega * sqrt(L*C)
alpha_lossless = S.Zero
step('Lossless (R\'=G\'=0): β =', beta_lossless)
step('v_p = 1/√(L\'C\') — independent of frequency: NON-DISPERSIVE')
step('Same as EM wave: v_p = 1/√(με) = c/√(ε_r μ_r)')

## §3 — Branch Cuts: Where √(ZY) Lives

In [ ]:
section('Branch cuts in γ = √(ZY)')

display(Markdown(r'''
γ = √(ZY) requires choosing a **branch** of the square root.

The square root $\sqrt{w}$ is double-valued: $w = re^{i\theta}$ gives
$$\sqrt{w} = \sqrt{r}\,e^{i\theta/2} \quad \text{or} \quad \sqrt{r}\,e^{i(\theta+2\pi)/2} = -\sqrt{r}\,e^{i\theta/2}$$

Branch cut convention: require $\text{Re}(\gamma) \geq 0$ (forward wave decays, not grows).
This fixes the physical branch.

**Where ZY = 0 or ∞** (branch points):
- $Z = 0$: $\omega = iR'/L'$ (complex frequency — not on real axis)
- $Y = 0$: $\omega = iG'/C'$ (complex frequency)
- These are in the upper half complex-ω plane → causal system (Kramers-Kronig again)
'''))

# Explicit γ for lossy line
w = symbols('w', complex=True)
Z_ex = 1 + I*w        # R'=1, L'=1
Y_ex = Rational(1,100) + I*w  # G'=0.01, C'=1
gamma_ex = sqrt(Z_ex * Y_ex)
step('Example: Z=1+iω, Y=0.01+iω')
step('γ(ω) = √(ZY) =', gamma_ex)

# At specific frequencies
for freq in [0, 1, 10, 100]:
    g = complex(gamma_ex.subs(w, freq).evalf())
    print(f'  ω={freq:4d}: γ = {g.real:.4f} + {g.imag:.4f}i  (α={g.real:.4f}, β={g.imag:.4f})')

display(Markdown(r'''
**High frequency limit** ($\omega \gg R'/L'$ and $\omega \gg G'/C'$):
$$\gamma \approx i\omega\sqrt{L'C'} + \frac{1}{2}\left(\frac{R'}{Z_0} + G'Z_0\right)$$
where $Z_0 = \sqrt{L'/C'}$.
Phase term $i\omega\sqrt{L'C'}$ is linear → non-dispersive at high frequency.
'''))

## §4 — Characteristic Impedance Z₀: Also Complex

In [ ]:
section('Characteristic impedance')

Z_0 = sqrt(Z_ser / Y_shu)
step('Characteristic impedance Z₀ = √(Z/Y) =', Z_0)

# Lossless
Z_0_lossless = sqrt(L/C)
step('Lossless: Z₀ = √(L\'/C\') =', Z_0_lossless, '  (real, purely resistive)')

# Common values
display(Markdown(r'''
| Line type | Z₀ | Use |
|---|---|---|
| Coax (RG-58) | 50 Ω | RF, lab instruments |
| Coax (RG-6) | 75 Ω | Cable TV |
| PCB microstrip | 50 Ω (designed) | high-speed digital |
| Twin-lead | 300 Ω | TV antenna |
| Optical fiber | n/a (Z₀=377Ω in free space) | photonics |
'''))

# Reflection coefficient
Z_L = symbols('Z_L', complex=True)
Gamma = (Z_L - Z_0_lossless) / (Z_L + Z_0_lossless)
step('Reflection coefficient Γ = (Z_L − Z₀)/(Z_L + Z₀) =', Gamma)
step('Z_L = Z₀: Γ =', simplify(Gamma.subs(Z_L, Z_0_lossless)), ' (matched, no reflection)')
step('Z_L = 0 (short): Γ =', simplify(Gamma.subs(Z_L, 0)))
step('Z_L = ∞ (open): Γ =', simplify(limit(Gamma, Z_L, oo)))

# Standing wave ratio
SWR = (1 + Abs(Gamma)) / (1 - Abs(Gamma))
step('SWR = (1+|Γ|)/(1−|Γ|) =', SWR)
step('SWR=1: perfect match.  SWR=∞: total reflection')

## §5 — Smith Chart: Polar Plot of Γ

In [ ]:
section('Smith chart = polar plot of complex Γ')

display(Markdown(r'''
The **Smith chart** is the complex Γ plane (unit disk, since $|\Gamma| \leq 1$).

Normalized load $z_L = Z_L/Z_0 = r + ix$:
$$\Gamma = \frac{z_L - 1}{z_L + 1}$$

Constant resistance circles:
$$\left|\Gamma - \frac{r}{r+1}\right| = \frac{1}{r+1}$$

Constant reactance arcs:
$$\left|\Gamma - \left(1 + \frac{i}{x}\right)\right| = \frac{1}{|x|}$$

**Moving along a lossless line** = rotating Γ on a circle centered at origin.
One full rotation = half wavelength (λ/2).

**Connection to branch cuts**: the Smith chart boundary $|\Gamma|=1$
is where Re(Z_L)=0 (lossless reactive load).
The branch cut of $\sqrt{ZY}$ maps to the boundary of the unit disk in Γ-space.
'''))

# Symbolic: r=1, x=1 (normalized)
r_n, x_n = symbols('r x', real=True)
z_n = r_n + I*x_n
Gamma_n = (z_n - 1)/(z_n + 1)
step('Γ(z_L) =', Gamma_n)
step('At z_L = 1+i (R=Z₀, X=Z₀): Γ =', simplify(Gamma_n.subs([(r_n,1),(x_n,1)])))
step('|Γ| =', Abs(Gamma_n.subs([(r_n,1),(x_n,1)])).evalf())

## §6 — Dispersion in Transmission Lines vs Optical Fiber

In [ ]:
section('Dispersion: circuits vs fiber')

display(Markdown(r'''
| Property | Transmission line | Optical fiber |
|---|---|---|
| Wave equation | Telegrapher's | Helmholtz |
| Propagation constant | $\gamma = \alpha + i\beta$ | $\tilde{k} = \alpha_{opt}/2 + i\beta$ |
| Phase constant | $\beta = \omega\sqrt{L'C'}$ (lossless) | $\beta = \omega n_r / c$ |
| Dispersion | $\beta(\omega)$ nonlinear → group delay varies | $\beta_2 = d^2\beta/d\omega^2$ (GVD) |
| Characteristic impedance | $Z_0 = \sqrt{L'/C'}$ | Wave impedance $\eta = \sqrt{\mu/\varepsilon}$ |
| Reflection | $\Gamma = (Z_L-Z_0)/(Z_L+Z_0)$ | Fresnel: $r = (n_1-n_2)/(n_1+n_2)$ |
| Branch cut | $\sqrt{ZY}$ in complex ω-plane | $\sqrt{1-\omega_p^2/\omega^2}$ in plasma |

**They are the same equation** in different unit systems.
The Fresnel reflection formula IS the transmission line reflection formula
with $Z_0 \to \eta = \eta_0/n$.
'''))

# Prove it
n1, n2, eta0 = symbols('n_1 n_2 eta_0', positive=True)
eta1 = eta0 / n1
eta2 = eta0 / n2
Gamma_TL = (eta2 - eta1) / (eta2 + eta1)
step('Γ = (η₂−η₁)/(η₂+η₁) =', simplify(Gamma_TL))
step('= (1/n₂ − 1/n₁)/(1/n₂ + 1/n₁) =')
step('=', simplify(Gamma_TL))
Fresnel_r = (n1 - n2)/(n1 + n2)
step('Fresnel r = (n₁−n₂)/(n₁+n₂) =', Fresnel_r)
step('Are they equal? Γ + r =', simplify(Gamma_TL + Fresnel_r))
step('= 0  ✓  (sign convention: Γ = −r, consistent with η=η₀/n)')

## §7 — Wavenumber Everywhere: One Table

In [ ]:
section('k across all of physics')

display(Markdown(r'''
| System | Wavenumber k | Dispersion relation | Branch cut when |
|---|---|---|---|
| Free EM (vacuum) | $k=\omega/c$ | $\omega=ck$ | never (always real) |
| EM in medium | $k=\omega n/c$ | $\omega=ck/n(\omega)$ | $n$ complex (absorption) |
| Transmission line | $\beta=\omega\sqrt{L'C'}$ | $\omega=\beta/\sqrt{L'C'}$ | lossy: $\gamma=\sqrt{ZY}$ |
| Plasma | $k=\frac{\omega}{c}\sqrt{1-\omega_p^2/\omega^2}$ | evanescent below $\omega_p$ | $\omega < \omega_p$ (imaginary k) |
| Quantum particle | $k=\sqrt{2mE}/\hbar$ | $E=\hbar^2k^2/2m$ | $E<0$: bound state (imaginary k) |
| Fiber (dispersive) | $k=k_0+k_1\delta\omega+\frac{1}{2}\beta_2\delta\omega^2$ | Taylor expansion | never (but phase wraps) |
| Acoustic in solid | $k=\omega/c_p$ or $\omega/c_s$ | two branches (P and S) | mode coupling at interfaces |

**Imaginary k** = evanescent wave = exponential decay instead of oscillation.
This is the branch cut being crossed: $\sqrt{negative} = i\sqrt{positive}$.

In a circuit: below plasma frequency, in a waveguide below cutoff,
in quantum mechanics in a classically forbidden region — all the same math.
'''))

# Evanescent wave
omega_p, omega_sym, c_sym = symbols('omega_p omega c', positive=True)
k_plasma = omega_sym/c_sym * sqrt(1 - omega_p**2/omega_sym**2)
step('Plasma k(ω) =', k_plasma)
step('Below plasma freq (ω<ω_p): k becomes imaginary')
k_below = k_plasma.subs(omega_sym, omega_p/2)
step('At ω=ω_p/2: k =', simplify(k_below))
step('= purely imaginary → evanescent, decays as e^(−|k|z)')

## §8 — PCB Trace as Transmission Line: Practical Numbers

In [ ]:
section('PCB microstrip')

display(Markdown(r'''
A PCB trace over a ground plane is a microstrip transmission line.
For $Z_0 = 50\,\Omega$ on FR4 ($\varepsilon_r \approx 4.3$):
$$w/h \approx 1.9 \quad \text{(trace width / board thickness)}$$
Typically: $h = 1.6$ mm → $w \approx 3$ mm.
'''))

import numpy as np

eps_r = 4.3       # FR4
c0 = 3e8          # m/s
eps_eff = (eps_r + 1)/2 + (eps_r - 1)/2 * 1/np.sqrt(1 + 12)  # approximate
v_p = c0 / np.sqrt(eps_eff)
print(f'FR4 ε_r = {eps_r}')
print(f'Effective ε_eff ≈ {eps_eff:.2f}')
print(f'Phase velocity v_p = c/√ε_eff = {v_p/c0:.3f}c = {v_p/1e8:.2f}×10⁸ m/s')

# Propagation delay per unit length
t_pd = 1/v_p * 1e9  # ns/m
print(f'Propagation delay: {t_pd*1e-3:.3f} ns/mm = {t_pd:.2f} ns/m')

# Length for quarter wave at 1 GHz
f_ghz = 1e9
lam = v_p / f_ghz
print(f'λ at 1 GHz: {lam*100:.1f} cm')
print(f'λ/4 at 1 GHz: {lam/4*100:.2f} cm  (quarter-wave transformer length)')

display(Markdown(r'''
**Quarter-wave transformer**: $Z_{in} = Z_0^2/Z_L$
Transforms any real impedance to any other real impedance.
Used to match 50Ω driver to 75Ω antenna: $Z_0 = \sqrt{50\times75} = 61.2\,\Omega$.

**At GHz frequencies**: PCB traces ARE transmission lines.
Ignore this → reflections, ringing, EMI, failed signal integrity.
The branch cut of $\gamma=\sqrt{ZY}$ is why your 1 GHz signal looks different
from your 100 MHz signal on the same trace.
'''))